# VRP 03 — Passive execution of the ATM straddle (both sides)

**Discussion:** `discussions/Volatility_Risk_Premium/009_claude_execution_and_conditional_short.md` · **Code:** `vrp_execution.py`

Can a limit order near the mid recover the spread that killed VRP 02 (buy at ask −27%, mid bound −14%)? Same hourly ATM entries as VRP 02.

- Each leg rests at $P_b=\text{mid}-q(\text{mid}-\text{bid})$ (buy) or $P_s=\text{mid}+q(\text{ask}-\text{mid})$ (sell) for $W$ minutes; an unfilled leg is crossed at $t+W$ ("passive-then-cross").
- **Fill proxy on 5m snapshots:** the opposite touch reaches $P$, or a trade prints at or through $P$ (volume or last price changed). Optimistic at this grid.
- **Primary:** $q=0.5$, $W=15$ min. Paired $\Delta R$ = policy − aggressive per entry; session bootstrap.
- Verdict per side: `PASSIVE_HELPS` / `PASSIVE_HURTS` / `INCONCLUSIVE`.

Environment: `VRP03_OFFLINE=1` (cached delivery prices only), `VRP03_N_BOOT`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import vrp_long_straddle as ls
VRP = PROJECT_ROOT / 'notebooks' / 'Volatility_Risk_Premium'
from data_engineering.modeling import vrp_execution as ve
OFFLINE = os.getenv('VRP03_OFFLINE', '').strip() == '1'
N_BOOT = int(os.getenv('VRP03_N_BOOT', str(ve.N_BOOT)))
OUT = VRP / '03'
OUT.mkdir(parents=True, exist_ok=True)
print('VRP 03', ve.VERSION, ve.selftest())
print(json.dumps(ve.registration(), indent=2))
T0 = time.time()

VRP 03 0.1.1 {'version': '0.1.1', 'ok': True}
{
  "version": "0.1.1",
  "q_levels": [
    0.0,
    0.5
  ],
  "windows_min": [
    5,
    15,
    30,
    60
  ],
  "primary": [
    0.5,
    15
  ],
  "n_boot": 4000,
  "seed": 9003,
  "policy": "passive-then-cross",
  "entries": "VRP 02 (hourly, TTE 2-20h, ATM)",
  "decision": "per side: session-bootstrap CI of paired Delta R (policy - aggressive) > 0 PASSIVE_HELPS, < 0 PASSIVE_HURTS"
}


In [2]:
def load_chain(cols_wanted, where_extra, hourly):
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        have = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
        need = [c for c in ls.CHAIN_COLUMNS if c != 'settlement_currency']
        miss = sorted(set(need) - have)
        if miss:
            raise RuntimeError(f'normalized_option_chain lacks {miss}')
        cols = [c for c in cols_wanted if c in have]
        grid = f'AND EXTRACT(minute FROM n.state_time) % {ls.ENTRY_GRID_MIN} = 0' if hourly else ''
        df = con.execute(f'''
            WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
            SELECT {', '.join('n.' + c for c in cols)}
            FROM normalized_option_chain n
            JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
            WHERE n.minutes_to_expiry BETWEEN 60 AND {ls.MAX_TTE_MIN} {grid} {where_extra}
        ''').df()
    finally:
        con.close()
    for c in cols_wanted:
        if c not in df:
            df[c] = None if c == 'settlement_currency' else np.nan
    return df

def settlement(st, cache_out, offline):
    sessions = sorted(st.session.unique())
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        idx = con.execute('SELECT state_time, underlying, futures_index_price FROM time_series_feature_state').df()
    finally:
        con.close()
    approx = ls.approx_settlement(idx, sessions)
    names = sorted({ls.index_name(u, c) for u, c in zip(st.underlying, st.settlement_currency)})
    seed = VRP / '02' / '02_delivery_prices.csv'
    if not cache_out.exists() and seed.exists():
        cache_out.write_text(seed.read_text())
    if offline:
        delivery = pd.read_csv(cache_out, dtype={'session': str}) if cache_out.exists() else pd.DataFrame(columns=['index_name', 'session', 'delivery_price'])
    else:
        delivery = ls.fetch_delivery_prices(names, cache_csv=cache_out)
    delivery = delivery[delivery.session.isin(sessions)]
    x = ls.attach_settlement(st, delivery, approx)
    print('settlement source:', x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict())
    return x


## 1. Snapshots, ticks, entries

In [3]:
chain = load_chain(list(ve.CHAIN_COLUMNS), f"AND abs(ln(n.strike / n.option_forward_price)) < 0.05 AND n.state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'", hourly=False)
print(f'chain rows {len(chain):,} | trade fields present: last_price={chain.last_price.notna().any()}, volume_24h={chain.volume_24h.notna().any()}')
snap = ve.prepare_snapshots(chain)
del chain
ticks = ve.family_ticks(snap)
display(ticks.to_frame())
st = ve.straddle_entries(snap)
st = settlement(st, OUT / '03_delivery_prices.csv', OFFLINE)
st = st[np.isfinite(st.S_T)].reset_index(drop=True)
print(f'entries {len(st):,} | sessions {st.session.nunique()} | {time.time()-T0:.0f}s')

chain rows 746,874 | trade fields present: last_price=True, volume_24h=True


,tick
AVAX|linear|USDC|USDC,0.00050
BTC|linear|USDC|USDC,5.00000
ETH|linear|USDC|USDC,0.20000
HYPE|linear|USDC|USDC,0.00200
SOL|linear|USDC|USDC,0.10000
TRX|linear|USDC|USDC,0.00005
XRP|linear|USDC|USDC,0.00050


settlement source: {'exchange': 140}
entries 2,515 | sessions 20 | 20s


## 2. Policy grid and primary decision

In [4]:
res = ve.evaluate(st, snap, ticks, n_boot=N_BOOT)
grid = res['grid']
display(grid.round(4))
print(json.dumps(res['decision'], indent=2, default=float))

,side,q,window_min,n_entries,n_used,n_sessions,R_aggressive_same_M,delta_bps_M,delta_bps_ci_low,delta_bps_ci_high,fill_rate_call,fill_rate_put,fill_rate_both,touch_share,markout_halfspreads_median,delta_M,delta_ci_low,delta_ci_high,delta_verdict,R_policy_M,R_policy_ci_low,R_policy_ci_high,R_both_filled_mean,adverse_selection_gap_mid
0,buy,0.0,5,2515,2515,20,-0.2664,-7.2569,-16.4878,-2.2904,0.2624,0.2708,0.0211,0.1074,-1.8000,-0.0142,-0.0178,-0.0108,PASSIVE_HURTS,-0.2805,-0.4048,-0.1379,-0.1063,0.0732
1,buy,0.0,15,2515,2515,20,-0.2664,-2.6271,-4.0322,-1.2949,0.4437,0.4736,0.1360,0.1074,-1.7029,-0.0091,-0.0162,-0.0017,PASSIVE_HURTS,-0.2755,-0.4013,-0.1295,-0.1255,0.0543
2,buy,0.0,30,2515,2514,20,-0.2663,-2.1168,-3.9659,-0.3927,0.5793,0.6139,0.3050,0.1074,-1.6000,-0.0058,-0.0177,0.0070,INCONCLUSIVE,-0.2721,-0.3963,-0.1237,-0.1793,0.0017
3,buy,0.0,60,2515,2508,20,-0.2659,-0.7929,-2.7064,1.0563,0.7165,0.7435,0.5006,0.1074,-1.5000,0.0020,-0.0092,0.0145,INCONCLUSIVE,-0.2639,-0.3887,-0.1157,-0.1792,0.0012
4,buy,0.5,5,2515,2515,20,-0.2664,-6.0939,-15.2302,-1.2084,0.1614,0.1765,0.0143,0.2803,-1.8571,-0.0077,-0.0108,-0.0045,PASSIVE_HURTS,-0.2740,-0.3997,-0.1295,-0.0803,0.0268
5,buy,0.5,15,2515,2515,20,-0.2664,-1.3598,-2.9027,0.1089,0.3225,0.3503,0.0755,0.2803,-1.6000,-0.0015,-0.0095,0.0080,INCONCLUSIVE,-0.2679,-0.3971,-0.1190,-0.0752,0.0551
6,buy,0.5,30,2515,2514,20,-0.2663,-0.5023,-2.4315,1.3275,0.4648,0.4930,0.1837,0.2803,-1.5455,0.0032,-0.0098,0.0185,INCONCLUSIVE,-0.2632,-0.3902,-0.1106,-0.1378,-0.0038
7,buy,0.5,60,2515,2508,20,-0.2659,1.0894,-0.9502,3.1404,0.6195,0.6437,0.3634,0.2803,-1.4697,0.0151,0.0017,0.0314,PASSIVE_HELPS,-0.2508,-0.3778,-0.0986,-0.1292,0.0025
8,buy,NaN,0,2515,2515,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BASELINE_AGGRESSIVE,-0.2664,-0.3918,-0.1226,NaN,NaN
9,sell,0.0,5,2515,2435,20,-0.0607,-2.7514,-3.2338,-2.2700,0.2068,0.2115,0.0020,0.1074,-0.8000,-0.0278,-0.0374,-0.0193,PASSIVE_HURTS,-0.0885,-0.2976,0.0898,-0.1006,-0.1850


{
  "buy": {
    "verdict": "INCONCLUSIVE",
    "delta_M": -0.0015115982154515074,
    "delta_ci": [
      -0.009508583283702046,
      0.00804551606480921
    ],
    "R_policy_M": -0.26787748986197746,
    "R_policy_ci": [
      -0.3971088872118724,
      -0.11902724691103289
    ],
    "fill_rate_both": 0.07554671968190854,
    "n_used": 2515.0,
    "n_sessions": 20.0,
    "R_aggressive_same_M": -0.266365891646526,
    "delta_bps_M": -1.3598185188404697,
    "delta_bps_ci": [
      -2.9027110655847976,
      0.10890692691705886
    ]
  },
  "sell": {
    "verdict": "PASSIVE_HURTS",
    "delta_M": -0.033319945232262865,
    "delta_ci": [
      -0.05167174922210103,
      -0.019104782040114093
    ],
    "R_policy_M": -0.0912531496406026,
    "R_policy_ci": [
      -0.3078220211224442,
      0.08877835314286879
    ],
    "fill_rate_both": 0.022266401590457258,
    "n_used": 2418.0,
    "n_sessions": 20.0,
    "R_aggressive_same_M": -0.057933204408339746,
    "delta_bps_M": -3.15877555

## 3. Primary configuration by TTE bucket and asset (reported)

In [5]:
rows = []
for side, pe in res['per_entry'].items():
    for by in ('tte_bucket', 'underlying'):
        for lvl, g in pe.groupby(by):
            d = ve.session_boot(g.delta.to_numpy(), g.session.to_numpy(), 1000)
            a = ve.session_boot(g.R_policy.to_numpy(), g.session.to_numpy(), 1000)
            rows.append({'side': side, 'group': by, 'level': lvl, 'n': len(g), 'fill_both': g.both_filled.mean(),
                         'delta_M': d.get('M'), 'delta_ci_low': d.get('ci_low'), 'delta_ci_high': d.get('ci_high'),
                         'R_policy_M': a.get('M'), 'R_ci_low': a.get('ci_low'), 'R_ci_high': a.get('ci_high')})
by = pd.DataFrame(rows)
display(by.round(4))

,side,group,level,n,fill_both,delta_M,delta_ci_low,delta_ci_high,R_policy_M,R_ci_low,R_ci_high
0,buy,tte_bucket,16-20h,535,0.1009,-0.0087,-0.0274,0.0071,-0.2481,-0.3859,-0.0896
1,buy,tte_bucket,2-4h,336,0.0833,0.0099,-0.0109,0.0284,-0.3118,-0.4856,-0.1066
2,buy,tte_bucket,4-8h,537,0.0521,-0.0019,-0.0142,0.0134,-0.2806,-0.4325,-0.1022
3,buy,tte_bucket,8-16h,1107,0.0723,-0.0027,-0.0076,0.0034,-0.2496,-0.4068,-0.0793
4,buy,underlying,AVAX,372,0.1237,-0.0298,-0.0571,-0.0017,0.0073,-0.2847,0.3863
5,buy,underlying,BTC,375,0.0507,-0.0122,-0.0208,-0.0041,-0.2936,-0.4814,-0.0851
6,buy,underlying,ETH,375,0.0480,-0.0091,-0.0208,0.0009,-0.2193,-0.4173,0.0695
7,buy,underlying,HYPE,361,0.0886,0.0121,-0.0031,0.0291,-0.3403,-0.4759,-0.1779
8,buy,underlying,SOL,367,0.0708,0.0232,0.0080,0.0405,-0.3380,-0.5002,-0.1565
9,buy,underlying,TRX,325,0.0277,-0.0048,-0.0146,0.0052,-0.4439,-0.5844,-0.2901


## 4. Outputs

In [6]:
grid.to_csv(OUT / '03_policy_grid.csv', index=False, float_format='%.6g')
by.to_csv(OUT / '03_primary_by_group.csv', index=False, float_format='%.6g')
ticks.to_frame().to_csv(OUT / '03_ticks.csv')
(OUT / '03_decision.json').write_text(json.dumps({'decision': res['decision'], 'registration': ve.registration()}, indent=2, default=float))
meta = {'module': '03_passive_execution', 'version': ve.VERSION, 'offline': OFFLINE, 'n_entries': int(len(st)),
        'sessions': [st.session.min(), st.session.max()], 'n_sessions': int(st.session.nunique()),
        'settle_sources': st.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict(),
        'runtime_s': round(time.time() - T0)}
(OUT / '03_run_metadata.json').write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str))

{
  "module": "03_passive_execution",
  "version": "0.1.1",
  "offline": false,
  "n_entries": 2515,
  "sessions": [
    "2026-09-21",
    "2026-10-10"
  ],
  "n_sessions": 20,
  "settle_sources": {
    "exchange": 140
  },
  "runtime_s": 36
}
